In [ ]:
import os
import glob
import pdfplumber
import pandas as pd
import numpy as np

# 1. Setup absolute paths and station map
pdf_folder = "/home/prasad/data/raw/cpcb"
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))

target_stations = {
    "10148": "Ganga at Har-Ki-Pauri Ghat",
    "1067":  "Ganga at Jajmau Bridge",
    "1049":  "Ganga at Sangam",
    "1071":  "Ganga at Malviya Bridge",
    "2552":  "Ganga at Gandhi Ghat"
}

extracted_records = []
print(f"Scanning {len(pdf_files)} target CPCB reports safely...\n")

# 2. Sequential file extraction loop
for i, pdf_path in enumerate(pdf_files, 1):
    filename = os.path.basename(pdf_path)
    print(f"[{i}/{len(pdf_files)}] Parsing: {filename}...", end="", flush=True)
    
    # Isolate year digits
    year_digits = "".join(filter(str.isdigit, filename))
    year = year_digits if len(year_digits) == 4 else "Unknown"
    
    file_rows = 0
    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Step through pages sequentially
            for page in pdf.pages:
                text = page.extract_text(layout=False) # Fast text layout retrieval
                if not text: 
                    continue
                
                text_upper = text.upper()
                # Fast pre-filtering boundary conditions
                if "GANGA" not in text_upper:
                    continue
                    
                lines = text.split("\n")
                for line in lines:
                    tokens = line.split()
                    if not tokens: 
                        continue
                    
                    # Target matching on the first index token element
                    station_code = tokens[0]
                    if station_code in target_stations:
                        extracted_records.append({
                            "station_code": station_code,
                            "station_name": target_stations[station_code],
                            "year": year,
                            "raw_line_text": line,
                            "file_source": filename
                        })
                        file_rows += 1
                        
        print(f" Done! Found {file_rows} records.")
    except Exception as e:
        print(f" Skinned/Error parsing pages: {e}")

# 3. Compile and save results
df_extracted = pd.DataFrame(extracted_records)
print(f"\nProcessing Complete! Total Extracted Rows: {len(df_extracted)}")

if not df_extracted.empty:
    processed_folder = "/home/prasad/data/processed"
    os.makedirs(processed_folder, exist_ok=True)
    df_extracted.to_csv(os.path.join(processed_folder, "wq_raw_extracted.csv"), index=False)
    print(f"Interim matrix written directly to: {processed_folder}/wq_raw_extracted.csv")
    display(df_extracted.head(5))


Scanning 14 target CPCB reports safely...

[1/14] Parsing: NWMP_DATA_2015.pdf... Done! Found 4 records.
[2/14] Parsing: WQuality_River-Data-2018.pdf... Done! Found 5 records.
[3/14] Parsing: WQuality_River-Data-2019.pdf... Done! Found 5 records.
[4/14] Parsing: WQuality_River-Data-2020.pdf... Done! Found 5 records.
[5/14] Parsing: WQuality_River-Data-2021.pdf... Done! Found 5 records.
[6/14] Parsing: WQuality_River-Data-2022.pdf... Done! Found 5 records.
[7/14] Parsing: WQuality_River-Data-2023.pdf...

In [ ]:
import os
import glob
import pdfplumber
import pandas as pd
import numpy as np

# 1. Setup absolute paths and target station configuration
pdf_folder = "/home/prasad/data/raw/cpcb"
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))

target_stations = {
    "10148": "Ganga at Har-Ki-Pauri Ghat",
    "1067":  "Ganga at Jajmau Bridge",
    "1049":  "Ganga at Sangam",
    "1071":  "Ganga at Malviya Bridge",
    "2552":  "Ganga at Gandhi Ghat"
}

extracted_records = []
print(f"Starting 100% complete scan for {len(pdf_files)} CPCB reports...\n")

# 2. Comprehensive page-by-page extraction loop
for i, pdf_path in enumerate(pdf_files, 1):
    filename = os.path.basename(pdf_path)
    print(f"[{i}/{len(pdf_files)}] Scanning ALL pages of: {filename}...")
    
    # FIXED: Added the required second argument 'filename' to the filter function
    year_digits = "".join(filter(str.isdigit, filename))
    year = year_digits if len(year_digits) == 4 else "Unknown"
    
    file_rows = 0
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                text = page.extract_text()
                if not text: 
                    continue
                    
                lines = text.split("\n")
                for line in lines:
                    for code, clean_name in target_stations.items():
                        if code in line:
                            extracted_records.append({
                                "station_code": code,
                                "station_name": clean_name,
                                "year": year,
                                "page_number": page_num,
                                "raw_line_text": line,
                                "file_source": filename
                            })
                            file_rows += 1
                            break
                            
        print(f"   -> Finished {filename}! Found {file_rows} records.\n")
    except Exception as e:
        print(f"   -> Error parsing {filename}: {e}\n")

# 3. Structural verification and clean compilation
df_extracted = pd.DataFrame(extracted_records)
if not df_extracted.empty:
    df_extracted = df_extracted.drop_duplicates(subset=["station_code", "year", "raw_line_text"])
    
print(f"\nScan Complete! Total verified rows captured: {len(df_extracted)}")

if not df_extracted.empty:
    processed_folder = "/home/prasad/data/processed"
    os.makedirs(processed_folder, exist_ok=True)
    df_extracted.to_csv(os.path.join(processed_folder, "wq_raw_extracted.csv"), index=False)
    print(f"Complete raw data matrix safely written to: {processed_folder}/wq_raw_extracted.csv")
    
    print("\nBreakdown of records captured per year:")
    display(df_extracted.groupby("year").size().reset_index(name="rows_captured"))


Starting 100% complete scan for 14 CPCB reports...

[1/14] Scanning ALL pages of: NWMP_DATA_2015.pdf...
   -> Finished NWMP_DATA_2015.pdf! Found 6 records.

[2/14] Scanning ALL pages of: WQuality_River-Data-2018.pdf...
   -> Finished WQuality_River-Data-2018.pdf! Found 5 records.

[3/14] Scanning ALL pages of: WQuality_River-Data-2019.pdf...
   -> Finished WQuality_River-Data-2019.pdf! Found 6 records.

[4/14] Scanning ALL pages of: WQuality_River-Data-2020.pdf...
   -> Finished WQuality_River-Data-2020.pdf! Found 5 records.

[5/14] Scanning ALL pages of: WQuality_River-Data-2021.pdf...
